# Code Review

> Notebook generated from [`examples/subgraphs/04_code_review.py`](../../examples/subgraphs/04_code_review.py).

| Item | Value |
|------|-------|
| **Dataset** | CodeSearchNet (embedded, Python snippets) |
| **API key** | ⚠️  **Requires API key** (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) in `.env`. |

**Expected result:** linter → security_scanner → logic_reviewer → suggester → report. Weighted score + final verdict.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
Code Review Subgraph — Automated Python code review
=============================================================
Subgraph: prismal.agents.subgraphs.code_review

Dataset: CodeSearchNet (Python) — real code fragments from GitHub
  • 2.3M Python functions labeled with docstrings (CodeSearchNet corpus).
  • Reference: https://huggingface.co/datasets/code_search_net
  • Why: The code_review subgraph has 5 specialized nodes that
    detect different kinds of problems. CodeSearchNet snippets
    are real code with security bugs (SQL injection, hardcoded
    credentials), logic errors (division by zero, out-of-range
    indices), and style problems (functions without docstring, magic numbers).
    Each category is detected by a different node of the pipeline.

Code Review subgraph description:
  linter → security_scanner → logic_reviewer → suggester → report_generator

  Nodes:
  1. linter           — PEP 8, cyclomatic complexity, naming, docstrings
  2. security_scanner — injection, hardcoded credentials, deserialization
  3. logic_reviewer   — division by zero, indices, unreachable conditions
  4. suggester        — generates remediation suggestions for each issue
  5. report_generator — severity-weighted score + approved/rejected flag

  Score = 1.0 - Σ severity_weight[issue]
  Weights: critical=0.4, high=0.2, medium=0.1, low=0.05, info=0.01
  approved = (score >= approval_threshold)  # default: 0.8

Usage:
    uv run python examples/subgraphs/04_code_review.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio
import re

from prismal.agents.subgraphs.code_review.types import CodeIssue

# Import with error handling in case the subgraph is not registered
try:
    from prismal.agents.subgraphs.code_review.builder import (
        build_code_review_subgraph,
        register_code_review,
    )
    from prismal.agents.subgraphs.factory import assemble_state_graph

    CODE_REVIEW_AVAILABLE = True
except ImportError:
    CODE_REVIEW_AVAILABLE = False

## Dataset: CodeSearchNet Python snippets with intentional issues

In [ ]:
# Real code from GitHub annotated with problems for each node of the pipeline.
CODE_SNIPPETS = [
    {
        "id": "CR-001",
        "filename": "db_utils.py",
        "description": "Query builder with SQL injection and no docstring",
        "expected_issues": ["security", "style"],
        "code": """\
import sqlite3
import pickle

DATABASE = "app.db"
ADMIN_PASSWORD = "supersecret123"   # hardcoded credential

def get_user(username):
    conn = sqlite3.connect(DATABASE)
    cursor = conn.cursor()
    # SQL injection: direct interpolation of user input
    query = "SELECT * FROM users WHERE username = '" + username + "'"
    cursor.execute(query)
    return cursor.fetchone()

def load_session(data):
    # Insecure deserialization of external data
    return pickle.loads(data)

def set_admin(user_id, is_admin):
    conn = sqlite3.connect(DATABASE)
    cursor = conn.cursor()
    cursor.execute(
        "UPDATE users SET is_admin=" + str(is_admin) + " WHERE id=" + str(user_id)
    )
    conn.commit()
""",
    },
    {
        "id": "CR-002",
        "filename": "math_ops.py",
        "description": "Math operations with logic errors",
        "expected_issues": ["logic", "style"],
        "code": """\
def calculate_average(numbers):
    \"\"\"Calculate the average of a list of numbers.\"\"\"
    total = sum(numbers)
    # ZeroDivisionError if numbers is empty
    return total / len(numbers)

def get_first_element(items):
    \"\"\"Get the first element without bounds checking.\"\"\"
    # IndexError if items is empty
    return items[0]

def classify_score(score):
    \"\"\"Classify a score into a grade.\"\"\"
    if score >= 90:
        return "A"
    elif score >= 80:
        return "B"
    elif score >= 70:
        return "C"
    elif score >= 60:
        return "D"
    elif score >= 60:   # unreachable condition (duplicated)
        return "E"
    else:
        return "F"

def compute_ratio(a, b):
    # No docstring + magic number
    return a / b * 3.14159265358979
""",
    },
    {
        "id": "CR-003",
        "filename": "file_handler.py",
        "description": "File handling with performance and security issues",
        "expected_issues": ["security", "performance", "style"],
        "code": """\
import os
import subprocess

SECRET_KEY = "my_very_secret_key_12345"
API_TOKEN  = "ghp_abc123verysecrettoken"

def read_large_file(filepath):
    # Performance: loads the whole file into memory
    with open(filepath) as f:
        return f.read()

def list_directory(path):
    # No path validation — path traversal
    return os.listdir(path)

def run_command(user_input):
    # Shell injection: shell=True with user input
    result = subprocess.run(user_input, shell=True, capture_output=True)
    return result.stdout.decode()

def process_files(directory):
    results = []
    for filename in os.listdir(directory):
        filepath = os.path.join(directory, filename)
        content = read_large_file(filepath)   # loads everything for each file
        results.append(content)
    return results
""",
    },
    {
        "id": "CR-004",
        "filename": "auth_service.py",
        "description": "Well-implemented authentication service (clean code)",
        "expected_issues": [],
        "code": """\
\"\"\"Authentication service with secure password handling.\"\"\"

from __future__ import annotations

import hashlib
import hmac
import os
import secrets
from typing import Optional


def hash_password(password: str, salt: bytes | None = None) -> tuple[str, bytes]:
    \"\"\"Hash a password with a random salt using PBKDF2-HMAC-SHA256.

    Args:
        password: The plaintext password to hash.
        salt: Optional salt bytes. A random 16-byte salt is generated if not provided.

    Returns:
        A tuple of (hashed_password_hex, salt_bytes).
    \"\"\"
    if salt is None:
        salt = os.urandom(16)
    key = hashlib.pbkdf2_hmac("sha256", password.encode("utf-8"), salt, 100_000)
    return key.hex(), salt


def verify_password(password: str, stored_hash: str, salt: bytes) -> bool:
    \"\"\"Verify a password against its stored hash.

    Uses constant-time comparison to prevent timing attacks.

    Args:
        password: The plaintext password to verify.
        stored_hash: The previously computed hash (hex string).
        salt: The salt used during hashing.

    Returns:
        True if the password matches the stored hash, False otherwise.
    \"\"\"
    candidate_hash, _ = hash_password(password, salt)
    return hmac.compare_digest(candidate_hash, stored_hash)


def generate_token(length: int = 32) -> str:
    \"\"\"Generate a cryptographically secure random token.

    Args:
        length: Number of random bytes (default 32 → 64 hex chars).

    Returns:
        A URL-safe base64 token string.
    \"\"\"
    return secrets.token_urlsafe(length)
""",
    },
    {
        "id": "CR-005",
        "filename": "data_pipeline.py",
        "description": "ETL Pipeline with mixed issues (varied severities)",
        "expected_issues": ["security", "logic", "performance", "style", "test"],
        "code": """\
import json
import yaml    # pyyaml can execute code with yaml.load()
import requests

API_KEY = "sk-prod-abc123secretkey"

def fetch_data(endpoint, params=None):
    # No timeout — may block indefinitely
    response = requests.get(endpoint, params=params)
    return response.json()

def parse_config(config_str):
    # yaml.load without Loader — arbitrary code execution
    return yaml.load(config_str)

def transform_records(records):
    \"\"\"Transform a list of records applying business rules.\"\"\"
    output = []
    for i in range(len(records)):
        record = records[i]
        # Access without key validation
        value = record["value"] * 2
        if value > 1000:
            output.append({"id": record["id"], "value": value, "flag": True})
        else:
            output.append({"id": record["id"], "value": value, "flag": False})
    return output

def save_results(data, filepath):
    # No error handling for I/O
    with open(filepath, "w") as f:
        json.dump(data, f)

# No unit tests for any function
""",
    },
]

## Severity weights for the score (same as report_generator_node)

In [ ]:
SEVERITY_WEIGHTS: dict[str, float] = {
    "critical": 0.40,
    "high": 0.20,
    "medium": 0.10,
    "low": 0.05,
    "info": 0.01,
}

## Injectable callables for the subgraph

In [ ]:
async def heuristic_linter(code: str, filename: str) -> list[CodeIssue]:
    """Heuristic linter: detects style and structure problems."""
    issues: list[CodeIssue] = []
    lines = code.splitlines()

    for i, line in enumerate(lines, start=1):
        stripped = line.strip()

        # Function without docstring
        if stripped.startswith("def ") and not stripped.startswith("def __"):
            # Check whether the next line is not a docstring
            next_line = lines[i].strip() if i < len(lines) else ""
            if not next_line.startswith('"""') and not next_line.startswith("'''"):
                func_name = stripped.split("(")[0].replace("def ", "")
                issues.append(
                    CodeIssue(
                        severity="low",
                        category="style",
                        description=f"Function `{func_name}` without docstring.",
                        file=filename,
                        line=i,
                        suggestion="Add a docstring with Args/Returns following Google style.",
                    )
                )

        # Magic numbers (exclude 0 and 1)
        magic = re.findall(r"\b(\d{2,}(?:\.\d+)?)\b", stripped)
        if magic and not stripped.startswith("#") and not stripped.startswith('"""'):
            for num in magic:
                if num not in ("10", "16", "32", "64", "128", "256"):
                    issues.append(
                        CodeIssue(
                            severity="info",
                            category="style",
                            description=f"Magic number `{num}` detected.",
                            file=filename,
                            line=i,
                            suggestion=f"Extract `{num}` to a constant with a descriptive name.",
                        )
                    )
                    break  # one issue per line is enough

        # Lines too long (>99 chars)
        if len(line) > 99:
            issues.append(
                CodeIssue(
                    severity="info",
                    category="style",
                    description=f"Line {i} exceeds 99 characters ({len(line)} chars).",
                    file=filename,
                    line=i,
                    suggestion="Split the expression or use parentheses to continue on the next line.",
                )
            )

    return issues


async def heuristic_scanner(code: str, filename: str) -> list[CodeIssue]:
    """Heuristic security scanner: detects vulnerability patterns."""
    issues: list[CodeIssue] = []
    lines = code.splitlines()

    patterns = [
        # SQL injection: string concatenation in queries
        (
            r"""(execute|cursor\.execute)\s*\(\s*["'][^"']*["']\s*\+""",
            "critical",
            "security",
            "SQL injection: direct interpolation of variables in query.",
            "Use parameterized queries: cursor.execute(sql, (param,))",
        ),
        # Hardcoded credentials
        (
            r"""(?i)(password|secret|api_key|token|credential)\s*=\s*["'][^"']{6,}["']""",
            "critical",
            "security",
            "Hardcoded credential in the source code.",
            "Use environment variables: os.environ.get('SECRET_KEY') or python-dotenv.",
        ),
        # pickle.loads
        (
            r"""pickle\.loads?\s*\(""",
            "high",
            "security",
            "Insecure deserialization with pickle: can execute arbitrary code.",
            "Use json.loads() for untrusted data or validate the origin with HMAC signing.",
        ),
        # yaml.load without Loader
        (
            r"""yaml\.load\s*\([^,)]+\)""",
            "high",
            "security",
            "yaml.load() without Loader can execute arbitrary code.",
            "Use yaml.safe_load() for untrusted data.",
        ),
        # subprocess with shell=True
        (
            r"""subprocess\.(run|call|Popen)\s*\([^)]*shell\s*=\s*True""",
            "high",
            "security",
            "subprocess with shell=True: vulnerable to shell injection.",
            "Pass arguments as a list: subprocess.run(['cmd', arg]) with shell=False.",
        ),
        # requests without timeout
        (
            r"""requests\.(get|post|put|delete|patch)\s*\([^)]*\)(?!\s*\.|\s*,\s*timeout)""",
            "medium",
            "security",
            "HTTP call without timeout: may block indefinitely.",
            "Add timeout=30: requests.get(url, timeout=30)",
        ),
    ]

    for pattern, severity, category, description, suggestion in patterns:
        for i, line in enumerate(lines, start=1):
            if re.search(pattern, line):
                issues.append(
                    CodeIssue(
                        severity=severity,  # type: ignore[arg-type]
                        category=category,  # type: ignore[arg-type]
                        description=description,
                        file=filename,
                        line=i,
                        suggestion=suggestion,
                    )
                )

    return issues


async def heuristic_reviewer(code: str, filename: str) -> list[CodeIssue]:
    """Heuristic logic reviewer: detects logic errors and conditions."""
    issues: list[CodeIssue] = []
    lines = code.splitlines()

    for i, line in enumerate(lines, start=1):
        stripped = line.strip()

        # Division without zero check
        if re.search(r"/\s*len\s*\(", stripped) and not re.search(
            r"if.*len", "".join(lines[max(0, i - 4) : i])
        ):
            issues.append(
                CodeIssue(
                    severity="high",
                    category="logic",
                    description="Division by len() without checking the collection is non-empty.",
                    file=filename,
                    line=i,
                    suggestion="Add: if not numbers: return 0.0  (or raise ValueError) before dividing.",
                )
            )

        # Index [0] access without check
        if re.search(r"\w+\[0\]", stripped) and not re.search(
            r"if\s+\w+|len\(", "".join(lines[max(0, i - 4) : i])
        ):
            issues.append(
                CodeIssue(
                    severity="medium",
                    category="logic",
                    description="Index [0] access without checking the collection is non-empty.",
                    file=filename,
                    line=i,
                    suggestion="Use: items[0] if items else None  or check len(items) > 0 first.",
                )
            )

        # Duplicated/unreachable condition in elif
        if stripped.startswith("elif "):
            cond = stripped[5:].split(":")[0].strip()
            # Look for the same condition in previous elifs of the same function
            prev_block = "\n".join(lines[max(0, i - 20) : i - 1])
            if f"elif {cond}" in prev_block or f"if {cond}" in prev_block:
                issues.append(
                    CodeIssue(
                        severity="medium",
                        category="logic",
                        description=f"Condition `elif {cond}` duplicated/unreachable.",
                        file=filename,
                        line=i,
                        suggestion="Remove the duplicated branch or fix the condition.",
                    )
                )

        # Loop for i in range(len(...)) — anti-pattern
        if re.search(r"for\s+\w+\s+in\s+range\s*\(\s*len\s*\(", stripped):
            issues.append(
                CodeIssue(
                    severity="low",
                    category="performance",
                    description="Loop `for i in range(len(x))` is less pythonic and efficient.",
                    file=filename,
                    line=i,
                    suggestion="Use `for item in items:` or `for i, item in enumerate(items):`",
                )
            )

        # Absence of tests (file without 'def test_' and without pytest/unittest import)
    if "def test_" not in code and "import pytest" not in code and "import unittest" not in code:
        if any(line.strip().startswith("def ") for line in lines):
            issues.append(
                CodeIssue(
                    severity="medium",
                    category="test",
                    description="No test functions detected for this module.",
                    file=filename,
                    line=None,
                    suggestion="Create a test_{filename} file with pytest and cover edge cases.",
                )
            )

    return issues


async def heuristic_suggester(code: str, issues: list[CodeIssue]) -> list[str]:
    """Generate summarized remediation suggestions per issue."""
    suggestions = []
    for issue in issues:
        loc = f"line {issue.line}" if issue.line else "whole module"
        suggestions.append(
            f"[{issue.severity.upper()}] {issue.file}:{loc} — {issue.description} "
            f"→ {issue.suggestion}"
        )
    return suggestions

## Score calculation (simulates report_generator)

In [ ]:
def compute_score(issues: list[CodeIssue]) -> float:
    """Calculate the review score (1.0 = no issues)."""
    penalty = sum(SEVERITY_WEIGHTS.get(issue.severity, 0.0) for issue in issues)
    return max(0.0, 1.0 - penalty)

## Run the review

In [ ]:
async def run_code_review(snippet: dict, approval_threshold: float = 0.8) -> dict:
    """Run the full review of a code snippet.

    Args:
        snippet: Dict with id, filename, description, code.
        approval_threshold: Minimum score to approve the PR.

    Returns:
        Dict with issues, score, approved, suggestions.
    """
    code = snippet["code"]
    filename = snippet["filename"]

    print(f"\n[{snippet['id']}] {snippet['filename']}")
    print(f"  Description: {snippet['description']}")

    if not CODE_REVIEW_AVAILABLE:
        # Demo mode: run the callables directly without the subgraph
        print("  [Demo mode — simulated subgraph]\n")

        # Node 1: linter
        print("  ── Node 1: linter ──")
        linter_issues = await heuristic_linter(code, filename)
        print(f"    Style issues: {len(linter_issues)}")

        # Node 2: security_scanner
        print("  ── Node 2: security_scanner ──")
        scanner_issues = await heuristic_scanner(code, filename)
        print(f"    Security issues: {len(scanner_issues)}")

        # Node 3: logic_reviewer
        print("  ── Node 3: logic_reviewer ──")
        logic_issues = await heuristic_reviewer(code, filename)
        print(f"    Logic/test issues: {len(logic_issues)}")

        # Consolidate issues
        all_issues = linter_issues + scanner_issues + logic_issues

        # Node 4: suggester
        print("  ── Node 4: suggester ──")
        suggestions = await heuristic_suggester(code, all_issues)
        print(f"    Suggestions generated: {len(suggestions)}")

        # Node 5: report_generator
        print("  ── Node 5: report_generator ──")
        score = compute_score(all_issues)
        approved = score >= approval_threshold
        verdict = "✓ APPROVED" if approved else "✗ REJECTED"
        print(f"    Score     : {score:.3f}")
        print(f"    Threshold : {approval_threshold}")
        print(f"    Verdict   : {verdict}")

        return {
            "id": snippet["id"],
            "filename": filename,
            "issues": all_issues,
            "score": score,
            "approved": approved,
            "suggestions": suggestions,
        }

    # Real mode with subgraph
    from langchain_core.messages import HumanMessage

    from prismal.agents.state import create_initial_state

    await register_code_review(approval_threshold=approval_threshold)
    subgraph_def = build_code_review_subgraph(
        linter_fn=heuristic_linter,
        scanner_fn=heuristic_scanner,
        reviewer_fn=heuristic_reviewer,
        suggester_fn=heuristic_suggester,
        approval_threshold=approval_threshold,
    )
    graph = assemble_state_graph(subgraph_def).compile()

    state = create_initial_state(session_id="nb-code-review")
    state["messages"] = [HumanMessage(content=f"Review the code of {filename}:\n\n{code}")]
    state["metadata"] = {
        "code_review": {
            "code": code,
            "filename": filename,
            "issues": [],
        }
    }

    config = {"configurable": {"thread_id": f"cr_{snippet['id']}_001"}}
    final_state = await graph.ainvoke(state, config=config)

    review_meta = final_state.get("metadata", {}).get("code_review", {})
    report = review_meta.get("report")
    if report:
        print(f"    Score    : {report.score:.3f}")
        print(f"    Verdict   : {'✓ APPROVED' if report.approved else '✗ REJECTED'}")

    return {
        "id": snippet["id"],
        "filename": filename,
        "issues": review_meta.get("issues", []),
        "score": report.score if report else 0.0,
        "approved": report.approved if report else False,
        "suggestions": review_meta.get("suggestions", []),
    }


def print_issue_table(issues: list[CodeIssue]) -> None:
    """Display issues as a table with visual indicators."""
    if not issues:
        print("    ✓ No issues detected")
        return

    severity_icons = {
        "critical": "🔴",
        "high": "🟠",
        "medium": "🟡",
        "low": "🔵",
        "info": "⚪",
    }
    category_labels = {
        "security": "SEC",
        "logic": "LOG",
        "style": "STY",
        "performance": "PERF",
        "test": "TEST",
    }

    for issue in issues:
        icon = severity_icons.get(issue.severity, "  ")
        cat = category_labels.get(issue.category, issue.category[:4].upper())
        loc = f"L{issue.line:3d}" if issue.line else "    "
        print(f"    {icon} [{issue.severity:8s}] [{cat:4s}] {loc}  {issue.description[:60]}")


async def main() -> None:
    APPROVAL_THRESHOLD = 0.8

    print("=" * 70)
    print("  Code Review Subgraph — Dataset: CodeSearchNet Python (GitHub)")
    print("=" * 70)

    # Subgraph architecture
    print("\n[Code Review subgraph architecture]")
    nodes = [
        ("linter          ", "PEP 8, cyclomatic complexity, docstrings, magic numbers"),
        ("security_scanner", "SQL injection, credentials, pickle, yaml.load, subprocess"),
        ("logic_reviewer  ", "ZeroDivision, IndexError, unreachable conditions"),
        ("suggester       ", "Generates remediations for each detected issue"),
        ("report_generator", "Severity-weighted score → approved / rejected"),
    ]
    print()
    for node, desc in nodes:
        print(f"  {node}: {desc}")
    print()
    print("  Score = 1.0 - Σ weight[severity]")
    print(
        f"  Weights: critical={SEVERITY_WEIGHTS['critical']}, high={SEVERITY_WEIGHTS['high']}, "
        f"medium={SEVERITY_WEIGHTS['medium']}, low={SEVERITY_WEIGHTS['low']}, info={SEVERITY_WEIGHTS['info']}"
    )
    print(f"  approved = (score >= {APPROVAL_THRESHOLD})")

    # Run reviews
    print(f"\n[Reviewing {len(CODE_SNIPPETS)} code snippets]")
    results = []
    for snippet in CODE_SNIPPETS:
        result = await run_code_review(snippet, approval_threshold=APPROVAL_THRESHOLD)
        results.append(result)

        # Show detailed issues
        print(f"\n  Detected issues ({len(result['issues'])} total):")
        print_issue_table(result["issues"])
        print("─" * 70)

    # ── Global statistics ──────────────────────────────────────────────────────
    print("\n[Statistical summary — all snippets]")

    all_issues: list[CodeIssue] = []
    for r in results:
        all_issues.extend(r["issues"])

    # Count per severity
    print("\n  Distribution by severity:")
    for sev in ("critical", "high", "medium", "low", "info"):
        count = sum(1 for i in all_issues if i.severity == sev)
        bar = "█" * count
        weight = SEVERITY_WEIGHTS[sev]
        print(f"    {sev:8s}  (×{weight:.2f})  {bar:<15} {count:2d}")

    # Count per category
    print("\n  Distribution by category:")
    for cat in ("security", "logic", "style", "performance", "test"):
        count = sum(1 for i in all_issues if i.category == cat)
        bar = "█" * count
        print(f"    {cat:12s}  {bar:<15} {count:2d}")

    # Results per file
    print("\n  Results per file:")
    print(f"  {'ID':<8} {'File':<22} {'Issues':>6} {'Score':>7} {'Verdict'}")
    print("  " + "─" * 60)
    for r in results:
        verdict = "✓ APPROVED" if r["approved"] else "✗ REJECTED"
        issue_count = len(r["issues"])
        print(f"  {r['id']:<8} {r['filename']:<22} {issue_count:>6} {r['score']:>7.3f}  {verdict}")

    total_approved = sum(1 for r in results if r["approved"])
    print(f"\n  Approved: {total_approved}/{len(results)}")
    print(f"  Total issues detected: {len(all_issues)}")

    # ── Severity comparison ────────────────────────────────────────────────────
    print("\n[Score impact by issue severity]")
    print(f"  A single critical issue subtracts {SEVERITY_WEIGHTS['critical']:.0%} from the score")
    print(
        f"  → With 1 critical: score={1.0 - SEVERITY_WEIGHTS['critical']:.2f}  "
        f"{'✓' if (1.0 - SEVERITY_WEIGHTS['critical']) >= APPROVAL_THRESHOLD else '✗'}"
    )
    print(
        f"  → With 2 high:     score={1.0 - 2 * SEVERITY_WEIGHTS['high']:.2f}  "
        f"{'✓' if (1.0 - 2 * SEVERITY_WEIGHTS['high']) >= APPROVAL_THRESHOLD else '✗'}"
    )
    print(
        f"  → With 4 medium:   score={1.0 - 4 * SEVERITY_WEIGHTS['medium']:.2f}  "
        f"{'✓' if (1.0 - 4 * SEVERITY_WEIGHTS['medium']) >= APPROVAL_THRESHOLD else '✗'}"
    )
    print(
        f"  → With 10 low:     score={1.0 - 10 * SEVERITY_WEIGHTS['low']:.2f}  "
        f"{'✓' if (1.0 - 10 * SEVERITY_WEIGHTS['low']) >= APPROVAL_THRESHOLD else '✗'}"
    )

    # ── Usage with custom callables ────────────────────────────────────────────
    print("\n[Usage with injectable callables]")
    print("  from prismal.agents.subgraphs.code_review.builder import (")
    print("      build_code_review_subgraph,")
    print("  )")
    print("  subgraph = build_code_review_subgraph(")
    print("      linter_fn=my_flake8_runner,     # async (code, filename) -> list[CodeIssue]")
    print("      scanner_fn=my_bandit_scanner,   # async (code, filename) -> list[CodeIssue]")
    print("      reviewer_fn=my_llm_reviewer,    # async (code, filename) -> list[CodeIssue]")
    print("      suggester_fn=my_llm_suggester,  # async (code, issues)   -> list[str]")
    print("      approval_threshold=0.85,")
    print("  )")
    print("  # If linter_fn=None, uses the default LLM from the ProviderRegistry")

    # ── Manual Code Review vs pipeline comparison ─────────────────────────────
    print("\n[Manual Code Review vs automated subgraph]")
    comparison = [
        ("Manual (1 reviewer)", "15-60 min", "Partial", "High", "High", "None"),
        ("Static linter", "< 1 s", "Style/PEP8", "Low", "Medium", "None"),
        ("Bandit / SAST", "< 5 s", "Security", "Low", "High", "None"),
        ("Code Review Subgraph", "5-30 s", "Complete", "High", "High", "Traceability"),
    ]
    header = (
        f"  {'Method':<26} {'Time':<11} {'Coverage':<13} {'Recall':<8} {'Precis.':<8} {'Extra'}"
    )
    print(header)
    print("  " + "─" * 75)
    for method, time_, coverage, recall, prec, extra in comparison:
        print(f"  {method:<26} {time_:<11} {coverage:<13} {recall:<8} {prec:<8} {extra}")


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()